In [1]:
   !ls /kaggle/input/notebooks/aadiparkhi
   !ls /kaggle/input/notebooks/aadiparkhi/*

01a2-bdd-curation  01a-data-curation
/kaggle/input/notebooks/aadiparkhi/01a2-bdd-curation:
bdd_class_names.json  custom.css  __notebook__.ipynb  __results__.html
bdd_manifest.csv      data	  __output__.json

/kaggle/input/notebooks/aadiparkhi/01a-data-curation:
bdd_manifest.csv  custom.css  __notebook__.ipynb  places_manifest.csv
class_names.json  data	      __output__.json	  __results__.html


In [2]:
import json, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

torch.manual_seed(42)
dev, EPOCHS = 'cuda', 10
NB = '/kaggle/input/notebooks/aadiparkhi'
PR, BR = f'{NB}/01a-data-curation/data', f'{NB}/01a2-bdd-curation/data'
pl = pd.read_csv(f'{NB}/01a-data-curation/places_manifest.csv')
bd = pd.read_csv(f'{NB}/01a2-bdd-curation/bdd_manifest.csv')
names = json.load(open(f'{NB}/01a2-bdd-curation/bdd_class_names.json'))['cond']
n1, n2 = pl.label.nunique(), bd.cond_id.nunique()

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tr_tf, ev_tf = T.Compose([T.RandomHorizontalFlip(), T.ToTensor(), norm]), T.Compose([T.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, df, root, cols, tf):
        self.p, self.y, self.tf = (root + '/' + df.path).tolist(), torch.tensor(df[cols].values), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

def loader(df, root, cols, split, bs):
    tr = split == 'train'
    return DataLoader(DS(df[df.split == split], root, cols, tr_tf if tr else ev_tf), batch_size=bs,
                      shuffle=tr, drop_last=tr, num_workers=4, pin_memory=True)

pt, bt = loader(pl, PR, ['label'], 'train', 64), loader(bd, BR, ['cond_id', 'person'], 'train', 32)
pv, bv = loader(pl, PR, ['label'], 'val', 128), loader(bd, BR, ['cond_id', 'person'], 'val', 64)
ps, bs = loader(pl, PR, ['label'], 'test', 128), loader(bd, BR, ['cond_id', 'person'], 'test', 64)

print(len(pl), len(bd), n1, n2, names)
xp, yp = next(iter(pt)); xb, yb = next(iter(bt))
print(xp.shape, yp.shape, xb.shape, yb.shape)

24000 13787 48 11 ['clear_day', 'clear_night', 'clear_twilight', 'overcast_day', 'overcast_twilight', 'rainy_day', 'rainy_night', 'rainy_twilight', 'snow_fog_day', 'snow_fog_night', 'snow_fog_twilight']
torch.Size([64, 3, 224, 224]) torch.Size([64, 1]) torch.Size([32, 3, 216, 384]) torch.Size([32, 2])


In [3]:
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        s.bb = models.resnet18(weights='IMAGENET1K_V1'); s.bb.fc = nn.Identity()
        s.h1, s.h2, s.h3 = nn.Linear(512, n1), nn.Linear(512, n2), nn.Linear(512, 2)
    def forward(s, x):
        z = s.bb(x)
        return s.h1(z), s.h2(z), s.h3(z)

@torch.no_grad()
def evaluate(pdl, bdl):
    model.eval()
    o = {k: [] for k in ['p1', 'y1', 'p2', 'y2', 'p3', 'y3']}
    for x, y in pdl:
        with torch.autocast('cuda'): a, _, _ = model(x.to(dev))
        o['p1'].append(a.argmax(1).cpu()); o['y1'].append(y[:, 0])
    for x, y in bdl:
        with torch.autocast('cuda'): _, b, c = model(x.to(dev))
        o['p2'].append(b.argmax(1).cpu()); o['y2'].append(y[:, 0])
        o['p3'].append(c.argmax(1).cpu()); o['y3'].append(y[:, 1])
    o = {k: torch.cat(v).numpy() for k, v in o.items()}
    m = dict(places=(o['p1'] == o['y1']).mean(), cond=(o['p2'] == o['y2']).mean(),
             person_bacc=balanced_accuracy_score(o['y3'], o['p3']), person_f1=f1_score(o['y3'], o['p3']))
    return m, o

def forever(d):
    while True:
        yield from d

model = Net().to(dev)
with torch.no_grad(), torch.autocast('cuda'):
    a, _, _ = model(xp.to(dev)); _, b, c = model(xb.to(dev))
print(a.shape, b.shape, c.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


torch.Size([64, 48]) torch.Size([32, 11]) torch.Size([32, 2])


In [4]:
cnt = bd[bd.split == 'train'].person.value_counts().sort_index().values
cew = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum() / (2 * cnt), dtype=torch.float).to(dev))
ce = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(bt))
scaler = torch.amp.GradScaler('cuda')
pi, best, hist = forever(pt), -1, []
for ep in range(EPOCHS):
    model.train()
    for m in model.modules():          # keep ImageNet BN stats: two domains would mix running stats
        if isinstance(m, nn.BatchNorm2d): m.eval()
    for xb, yb in bt:
        xp, yp = next(pi)
        xp, yp, xb, yb = xp.to(dev), yp[:, 0].to(dev), xb.to(dev), yb.to(dev)
        with torch.autocast('cuda'):
            a, _, _ = model(xp); _, b, c = model(xb)
            loss = ce(a.float(), yp) + ce(b.float(), yb[:, 0]) + cew(c.float(), yb[:, 1])
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    m, _ = evaluate(pv, bv)
    hist.append({'epoch': ep + 1, 'loss': loss.item(), **m}); print(hist[-1])
    score = m['places'] + m['cond'] + m['person_bacc']
    if score > best:
        best = score; torch.save(model.state_dict(), '/kaggle/working/multitask_resnet18.pt')
pd.DataFrame(hist).to_csv('/kaggle/working/train_history.csv', index=False)

/tmp/ipykernel_49/1473073178.py:18: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


{'epoch': 1, 'loss': 4.0151472091674805, 'places': np.float64(0.2679166666666667), 'cond': np.float64(0.5178571428571429), 'person_bacc': np.float64(0.7321996185632549), 'person_f1': 0.4763271162123386}
{'epoch': 2, 'loss': 3.366518974304199, 'places': np.float64(0.4479166666666667), 'cond': np.float64(0.6258503401360545), 'person_bacc': np.float64(0.7803758741258742), 'person_f1': 0.5411392405063291}
{'epoch': 3, 'loss': 2.5437817573547363, 'places': np.float64(0.5229166666666667), 'cond': np.float64(0.6717687074829932), 'person_bacc': np.float64(0.7886800699300699), 'person_f1': 0.639080459770115}
{'epoch': 4, 'loss': 2.709502935409546, 'places': np.float64(0.5695833333333333), 'cond': np.float64(0.6836734693877551), 'person_bacc': np.float64(0.8288103941513032), 'person_f1': 0.6612576064908722}
{'epoch': 5, 'loss': 1.641315221786499, 'places': np.float64(0.60125), 'cond': np.float64(0.6981292517006803), 'person_bacc': np.float64(0.7986530514939606), 'person_f1': 0.6604215456674473}


In [5]:
model.load_state_dict(torch.load('/kaggle/working/multitask_resnet18.pt'))
m, o = evaluate(ps, bs)
print('TEST', m)
g = pl[pl.split == 'test'].group.values
print('places acc by super-group (indoor, natural, man-made):', [round((o['p1'][g == i] == o['y1'][g == i]).mean(), 3) for i in range(3)])
print(classification_report(o['y2'], o['p2'], labels=range(n2), target_names=names, zero_division=0))
print(classification_report(o['y3'], o['p3'], target_names=['no_person', 'person']))
json.dump({k: float(v) for k, v in m.items()}, open('/kaggle/working/test_results.json', 'w'))

TEST {'places': np.float64(0.60125), 'cond': np.float64(0.6938250428816467), 'person_bacc': np.float64(0.8150289290353052), 'person_f1': 0.6804979253112033}
places acc by super-group (indoor, natural, man-made): [np.float64(0.634), np.float64(0.566), np.float64(0.604)]
                   precision    recall  f1-score   support

        clear_day       0.75      0.69      0.72       127
      clear_night       0.72      0.79      0.75       127
   clear_twilight       0.66      0.75      0.70       127
     overcast_day       0.61      0.66      0.63       127
overcast_twilight       0.56      0.48      0.52        88
        rainy_day       0.70      0.64      0.67       127
      rainy_night       0.79      0.70      0.74       127
   rainy_twilight       0.40      0.22      0.29        27
     snow_fog_day       0.68      0.79      0.73       127
   snow_fog_night       0.81      0.84      0.83       127
snow_fog_twilight       0.57      0.49      0.52        35

         accuracy   

In [6]:
tf_p = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1)), T.RandomHorizontalFlip(), T.ToTensor(), norm])
tf_b = T.Compose([T.RandomResizedCrop((216, 384), scale=(0.7, 1), ratio=(1.6, 1.9)), T.RandomHorizontalFlip(), T.ToTensor(), norm])

def loader2(df, root, cols, tf, bs):
    return DataLoader(DS(df[df.split == 'train'], root, cols, tf), batch_size=bs, shuffle=True,
                      drop_last=True, num_workers=4, pin_memory=True)

pt, bt = loader2(pl, PR, ['label'], tf_p, 64), loader2(bd, BR, ['cond_id', 'person'], tf_b, 32)

class Net50(nn.Module):
    def __init__(s):
        super().__init__()
        s.bb = models.resnet50(weights='IMAGENET1K_V2'); s.bb.fc = nn.Identity()
        s.h1, s.h2, s.h3 = nn.Linear(2048, n1), nn.Linear(2048, n2), nn.Linear(2048, 2)
    def forward(s, x):
        z = s.bb(x)
        return s.h1(z), s.h2(z), s.h3(z)

model = Net50().to(dev)
xp, yp = next(iter(pt)); xb, yb = next(iter(bt))
with torch.no_grad(), torch.autocast('cuda'):
    a, _, _ = model(xp.to(dev)); _, b, c = model(xb.to(dev))
print(xp.shape, xb.shape, a.shape, b.shape, c.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


torch.Size([64, 3, 224, 224]) torch.Size([32, 3, 216, 384]) torch.Size([64, 48]) torch.Size([32, 11]) torch.Size([32, 2])


In [7]:
EPOCHS = 12
cnt = bd[bd.split == 'train'].person.value_counts().sort_index().values
cew = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum() / (2 * cnt), dtype=torch.float).to(dev))
ce = nn.CrossEntropyLoss()
heads = [p for n, p in model.named_parameters() if n.startswith('h')]
opt = torch.optim.AdamW([{'params': model.bb.parameters(), 'lr': 1e-4}, {'params': heads, 'lr': 1e-3}], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(bt))
scaler = torch.amp.GradScaler('cuda')
pi, best, hist = forever(pt), -1, []
for ep in range(EPOCHS):
    model.train()
    for m in model.modules():          # keep ImageNet BN stats
        if isinstance(m, nn.BatchNorm2d): m.eval()
    for xb, yb in bt:
        xp, yp = next(pi)
        opt.zero_grad()
        with torch.autocast('cuda'):
            a, _, _ = model(xp.to(dev)); l1 = ce(a.float(), yp[:, 0].to(dev))
        scaler.scale(l1).backward()
        with torch.autocast('cuda'):
            _, b, c = model(xb.to(dev)); yb = yb.to(dev)
            l2 = ce(b.float(), yb[:, 0]) + cew(c.float(), yb[:, 1])
        scaler.scale(l2).backward()
        scaler.step(opt); scaler.update(); sched.step()
    m, _ = evaluate(pv, bv)
    hist.append({'epoch': ep + 1, 'loss': (l1 + l2).item(), **m}); print(hist[-1])
    score = m['places'] + m['cond'] + m['person_bacc']
    if score > best:
        best = score; torch.save(model.state_dict(), '/kaggle/working/multitask_resnet50.pt')
pd.DataFrame(hist).to_csv('/kaggle/working/train_history_r50.csv', index=False)

{'epoch': 1, 'loss': 2.038482904434204, 'places': np.float64(0.6725), 'cond': np.float64(0.689625850340136), 'person_bacc': np.float64(0.7973021296884933), 'person_f1': 0.6732186732186732}
{'epoch': 2, 'loss': 2.2542638778686523, 'places': np.float64(0.7079166666666666), 'cond': np.float64(0.7202380952380952), 'person_bacc': np.float64(0.8558685632549269), 'person_f1': 0.7607655502392344}
{'epoch': 3, 'loss': 2.2881875038146973, 'places': np.float64(0.6920833333333334), 'cond': np.float64(0.7074829931972789), 'person_bacc': np.float64(0.8846948506039416), 'person_f1': 0.7785234899328859}
{'epoch': 4, 'loss': 1.4169741868972778, 'places': np.float64(0.6966666666666667), 'cond': np.float64(0.701530612244898), 'person_bacc': np.float64(0.8718014939605849), 'person_f1': 0.7516483516483516}
{'epoch': 5, 'loss': 1.0328590869903564, 'places': np.float64(0.7066666666666667), 'cond': np.float64(0.7295918367346939), 'person_bacc': np.float64(0.875794659885569), 'person_f1': 0.7923627684964201}
{

In [8]:
model.load_state_dict(torch.load('/kaggle/working/multitask_resnet50.pt'))
m, o = evaluate(ps, bs)
print('TEST', m)
g = pl[pl.split == 'test'].group.values
print('places acc by super-group (indoor, natural, man-made):', [round((o['p1'][g == i] == o['y1'][g == i]).mean(), 3) for i in range(3)])
print(classification_report(o['y2'], o['p2'], labels=range(n2), target_names=names, zero_division=0))
print(classification_report(o['y3'], o['p3'], target_names=['no_person', 'person']))
json.dump({k: float(v) for k, v in m.items()}, open('/kaggle/working/test_results_r50.json', 'w'))


TEST {'places': np.float64(0.7029166666666666), 'cond': np.float64(0.7049742710120068), 'person_bacc': np.float64(0.8605337111819578), 'person_f1': 0.781038374717833}
places acc by super-group (indoor, natural, man-made): [np.float64(0.77), np.float64(0.584), np.float64(0.755)]
                   precision    recall  f1-score   support

        clear_day       0.76      0.69      0.72       127
      clear_night       0.70      0.88      0.78       127
   clear_twilight       0.66      0.64      0.65       127
     overcast_day       0.53      0.74      0.62       127
overcast_twilight       0.50      0.59      0.54        88
        rainy_day       0.85      0.65      0.73       127
      rainy_night       0.82      0.71      0.76       127
   rainy_twilight       0.43      0.11      0.18        27
     snow_fog_day       0.76      0.79      0.78       127
   snow_fog_night       0.88      0.82      0.85       127
snow_fog_twilight       0.65      0.49      0.56        35

         ac

In [9]:
class NetCX(nn.Module):
    def __init__(s):
        super().__init__()
        s.bb = models.convnext_tiny(weights='IMAGENET1K_V1'); s.bb.classifier[2] = nn.Identity()
        s.h1, s.h2, s.h3 = nn.Linear(768, n1), nn.Linear(768, n2), nn.Linear(768, 2)
    def forward(s, x):
        z = s.bb(x)
        return s.h1(z), s.h2(z), s.h3(z)

del model; torch.cuda.empty_cache()
model = NetCX().to(dev)
xp, yp = next(iter(pt)); xb, yb = next(iter(bt))
with torch.no_grad(), torch.autocast('cuda'):
    a, _, _ = model(xp.to(dev)); _, b, c = model(xb.to(dev))
print(a.shape, b.shape, c.shape)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 215MB/s] 


torch.Size([64, 48]) torch.Size([32, 11]) torch.Size([32, 2])


In [10]:
EPOCHS = 12
heads = [p for n, p in model.named_parameters() if n.startswith('h')]
opt = torch.optim.AdamW([{'params': model.bb.parameters(), 'lr': 5e-5}, {'params': heads, 'lr': 1e-3}], weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(bt))
scaler = torch.amp.GradScaler('cuda')
pi, best, hist = forever(pt), -1, []
for ep in range(EPOCHS):
    model.train()
    for xb, yb in bt:
        xp, yp = next(pi)
        opt.zero_grad()
        with torch.autocast('cuda'):
            a, _, _ = model(xp.to(dev)); l1 = ce(a.float(), yp[:, 0].to(dev))
        scaler.scale(l1).backward()
        with torch.autocast('cuda'):
            _, b, c = model(xb.to(dev)); yb = yb.to(dev)
            l2 = ce(b.float(), yb[:, 0]) + cew(c.float(), yb[:, 1])
        scaler.scale(l2).backward()
        scaler.step(opt); scaler.update(); sched.step()
    m, _ = evaluate(pv, bv)
    hist.append({'epoch': ep + 1, 'loss': (l1 + l2).item(), **m}); print(hist[-1])
    score = m['places'] + m['cond'] + m['person_bacc']
    if score > best:
        best = score; torch.save(model.state_dict(), '/kaggle/working/multitask_convnext.pt')
pd.DataFrame(hist).to_csv('/kaggle/working/train_history_convnext.csv', index=False)

{'epoch': 1, 'loss': 1.9189884662628174, 'places': np.float64(0.75125), 'cond': np.float64(0.7210884353741497), 'person_bacc': np.float64(0.8577558804831532), 'person_f1': 0.656084656084656}
{'epoch': 2, 'loss': 1.3411955833435059, 'places': np.float64(0.7516666666666667), 'cond': np.float64(0.7210884353741497), 'person_bacc': np.float64(0.8779799745708836), 'person_f1': 0.7565217391304347}
{'epoch': 3, 'loss': 1.2982169389724731, 'places': np.float64(0.7595833333333334), 'cond': np.float64(0.7355442176870748), 'person_bacc': np.float64(0.8877344246662429), 'person_f1': 0.7439024390243902}
{'epoch': 4, 'loss': 1.183018684387207, 'places': np.float64(0.7633333333333333), 'cond': np.float64(0.7159863945578231), 'person_bacc': np.float64(0.870311506675143), 'person_f1': 0.7960687960687961}
{'epoch': 5, 'loss': 1.0616384744644165, 'places': np.float64(0.7604166666666666), 'cond': np.float64(0.7227891156462585), 'person_bacc': np.float64(0.8802050222504768), 'person_f1': 0.7537473233404711}

In [11]:
model.load_state_dict(torch.load('/kaggle/working/multitask_convnext.pt'))
m, o = evaluate(ps, bs)
print('TEST', m)
g = pl[pl.split == 'test'].group.values
print('places acc by super-group (indoor, natural, man-made):', [round((o['p1'][g == i] == o['y1'][g == i]).mean(), 3) for i in range(3)])
print(classification_report(o['y2'], o['p2'], labels=range(n2), target_names=names, zero_division=0))
print(classification_report(o['y3'], o['p3'], target_names=['no_person', 'person']))
json.dump({k: float(v) for k, v in m.items()}, open('/kaggle/working/test_results_convnext.json', 'w'))

TEST {'places': np.float64(0.76375), 'cond': np.float64(0.7066895368782161), 'person_bacc': np.float64(0.8870563230605739), 'person_f1': 0.7941176470588235}
places acc by super-group (indoor, natural, man-made): [np.float64(0.829), np.float64(0.665), np.float64(0.798)]
                   precision    recall  f1-score   support

        clear_day       0.73      0.68      0.70       127
      clear_night       0.70      0.78      0.74       127
   clear_twilight       0.67      0.59      0.63       127
     overcast_day       0.62      0.70      0.66       127
overcast_twilight       0.52      0.50      0.51        88
        rainy_day       0.73      0.80      0.77       127
      rainy_night       0.81      0.72      0.76       127
   rainy_twilight       0.36      0.15      0.21        27
     snow_fog_day       0.72      0.83      0.77       127
   snow_fog_night       0.85      0.87      0.86       127
snow_fog_twilight       0.66      0.54      0.59        35

         accuracy   

In [12]:
from transformers import AutoModel

class NetDino(nn.Module):
    def __init__(s):
        super().__init__()
        s.bb = AutoModel.from_pretrained('facebook/dinov2-small')
        s.h1, s.h2, s.h3 = nn.Linear(768, n1), nn.Linear(768, n2), nn.Linear(768, 2)
    def forward(s, x):
        h = s.bb(pixel_values=x).last_hidden_state
        z = torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
        return s.h1(z), s.h2(z), s.h3(z)

del model; torch.cuda.empty_cache()
model = NetDino().to(dev)
xp, yp = next(iter(pt)); xb, yb = next(iter(bt))
with torch.no_grad(), torch.autocast('cuda'):
    a, _, _ = model(xp.to(dev)); _, b, c = model(xb.to(dev))
print(a.shape, b.shape, c.shape)

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

torch.Size([64, 48]) torch.Size([32, 11]) torch.Size([32, 2])


In [13]:
EPOCHS = 12
heads = [p for n, p in model.named_parameters() if n.startswith('h')]
opt = torch.optim.AdamW([{'params': model.bb.parameters(), 'lr': 2e-5}, {'params': heads, 'lr': 1e-3}], weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(bt))
scaler = torch.amp.GradScaler('cuda')
pi, best, hist = forever(pt), -1, []
for ep in range(EPOCHS):
    model.train()
    for xb, yb in bt:
        xp, yp = next(pi)
        opt.zero_grad()
        with torch.autocast('cuda'):
            a, _, _ = model(xp.to(dev)); l1 = ce(a.float(), yp[:, 0].to(dev))
        scaler.scale(l1).backward()
        with torch.autocast('cuda'):
            _, b, c = model(xb.to(dev)); yb = yb.to(dev)
            l2 = ce(b.float(), yb[:, 0]) + cew(c.float(), yb[:, 1])
        scaler.scale(l2).backward()
        scaler.step(opt); scaler.update(); sched.step()
    m, _ = evaluate(pv, bv)
    hist.append({'epoch': ep + 1, 'loss': (l1 + l2).item(), **m}); print(hist[-1])
    score = m['places'] + m['cond'] + m['person_bacc']
    if score > best:
        best = score; torch.save(model.state_dict(), '/kaggle/working/multitask_dino.pt')
pd.DataFrame(hist).to_csv('/kaggle/working/train_history_dino.csv', index=False)

/tmp/ipykernel_49/812404364.py:19: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


{'epoch': 1, 'loss': 1.9880787134170532, 'places': np.float64(0.74375), 'cond': np.float64(0.6785714285714286), 'person_bacc': np.float64(0.8478027654164018), 'person_f1': 0.754257907542579}
{'epoch': 2, 'loss': 1.9969251155853271, 'places': np.float64(0.7654166666666666), 'cond': np.float64(0.7210884353741497), 'person_bacc': np.float64(0.8620669103623648), 'person_f1': 0.7832512315270936}
{'epoch': 3, 'loss': 1.0941979885101318, 'places': np.float64(0.7666666666666667), 'cond': np.float64(0.7219387755102041), 'person_bacc': np.float64(0.8634376986649714), 'person_f1': 0.78239608801956}
{'epoch': 4, 'loss': 1.2901297807693481, 'places': np.float64(0.76), 'cond': np.float64(0.7346938775510204), 'person_bacc': np.float64(0.8925619834710744), 'person_f1': 0.7631027253668763}
{'epoch': 5, 'loss': 1.4745395183563232, 'places': np.float64(0.7595833333333334), 'cond': np.float64(0.7448979591836735), 'person_bacc': np.float64(0.8768078512396694), 'person_f1': 0.7788018433179723}
{'epoch': 6, 

In [14]:
model.load_state_dict(torch.load('/kaggle/working/multitask_dino.pt'))
m, o = evaluate(ps, bs)
print('TEST', m)
g = pl[pl.split == 'test'].group.values
print('places acc by super-group (indoor, natural, man-made):', [round((o['p1'][g == i] == o['y1'][g == i]).mean(), 3) for i in range(3)])
print(classification_report(o['y2'], o['p2'], labels=range(n2), target_names=names, zero_division=0))
print(classification_report(o['y3'], o['p3'], target_names=['no_person', 'person']))
json.dump({k: float(v) for k, v in m.items()}, open('/kaggle/working/test_results_dino.json', 'w'))

TEST {'places': np.float64(0.76875), 'cond': np.float64(0.7281303602058319), 'person_bacc': np.float64(0.8847845082063999), 'person_f1': 0.8198198198198198}
places acc by super-group (indoor, natural, man-made): [np.float64(0.848), np.float64(0.65), np.float64(0.809)]
                   precision    recall  f1-score   support

        clear_day       0.69      0.75      0.72       127
      clear_night       0.72      0.84      0.78       127
   clear_twilight       0.69      0.66      0.68       127
     overcast_day       0.59      0.68      0.63       127
overcast_twilight       0.61      0.59      0.60        88
        rainy_day       0.75      0.75      0.75       127
      rainy_night       0.79      0.72      0.76       127
   rainy_twilight       0.55      0.22      0.32        27
     snow_fog_day       0.80      0.79      0.79       127
   snow_fog_night       0.91      0.84      0.88       127
snow_fog_twilight       0.71      0.71      0.71        35

         accuracy    

In [15]:
!ls -lh /kaggle/working
!cd /kaggle/working && zip -j checkpoints.zip *.pt *.csv *.json

total 325M
-rw-r--r-- 1 root root 107M Oct 10 11:24 multitask_convnext.pt
-rw-r--r-- 1 root root  85M Oct 10 13:09 multitask_dino.pt
-rw-r--r-- 1 root root  43M Oct 10 10:13 multitask_resnet18.pt
-rw-r--r-- 1 root root  91M Oct 10 10:31 multitask_resnet50.pt
-rw-r--r-- 1 root root  115 Oct 10 12:07 test_results_convnext.json
-rw-r--r-- 1 root root  115 Oct 10 13:11 test_results_dino.json
-rw-r--r-- 1 root root  115 Oct 10 10:15 test_results.json
-rw-r--r-- 1 root root  125 Oct 10 11:02 test_results_r50.json
-rw-r--r-- 1 root root 1.2K Oct 10 11:44 train_history_convnext.csv
-rw-r--r-- 1 root root  988 Oct 10 10:15 train_history.csv
-rw-r--r-- 1 root root 1.2K Oct 10 13:09 train_history_dino.csv
-rw-r--r-- 1 root root 1.2K Oct 10 10:51 train_history_r50.csv
  adding: multitask_convnext.pt (deflated 7%)
  adding: multitask_dino.pt (deflated 7%)
  adding: multitask_resnet18.pt (deflated 7%)
  adding: multitask_resnet50.pt (deflated 7%)
  adding: train_history_convnext.csv (deflated 56%)
 